Preprocessing for meta parameter

In [24]:
# %% Zelle 1: Alle Importe
import os
import shutil
import queue
import threading
import time
import dicom_to_zarr
import helpers 
import dask.array as da
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import zarr
import tqdm
from torch.utils.data import Dataset, DataLoader

In [25]:
torch.cuda.is_available()

True

In [13]:
DATASET_NAME = '159269_B1'
PART = '14'

In [14]:
# Define paths (please adjust if needed)
dicom_folder = f'/mnt/dev_rep/repos/MRI-MoCoCo/ready_to_use/Raw_Datasets/{DATASET_NAME}/{PART}/DICOM' 
zarr_file = f'/mnt/dev_rep/repos/MRI-MoCoCo/ready_to_use/Raw_Datasets/{DATASET_NAME}/{PART}/{DATASET_NAME}_{PART}.zarr'
file_path = f'/mnt/dev_rep/repos/MRI-MoCoCo/ready_to_use/Raw_Datasets/{DATASET_NAME}/{PART}/AIF_2.txt'
results_path = f'/mnt/dev_rep/repos/MRI-MoCoCo/ready_to_use/MRI-Datasets/mdreg_motion_correction_results/{DATASET_NAME}_{PART}/'

In [15]:
print(dicom_folder)
print(zarr_file)
print(file_path)
print(results_path)

/mnt/dev_rep/repos/MRI-MoCoCo/ready_to_use/Raw_Datasets/159269_B1/14/DICOM
/mnt/dev_rep/repos/MRI-MoCoCo/ready_to_use/Raw_Datasets/159269_B1/14/159269_B1_14.zarr
/mnt/dev_rep/repos/MRI-MoCoCo/ready_to_use/Raw_Datasets/159269_B1/14/AIF_2.txt
/mnt/dev_rep/repos/MRI-MoCoCo/ready_to_use/MRI-Datasets/mdreg_motion_correction_results/159269_B1_14/


In [16]:
try:
    # Lade die erste (Index 0) und zweite (Index 1) Spalte
    tacq, aif = np.loadtxt(
        file_path, 
        skiprows=3,       
        usecols=(0, 1),   
        unpack=True         
    )

    print("Daten erfolgreich geladen!")
    print(f"Form des AIF-Arrays: {aif.shape}")
    print(f"Form des Zeit-Arrays: {tacq.shape}")
    print("\nErste 5 Werte des AIF:")
    print(aif[:5])
    print("\nErste 5 Werte der Zeitachse (in Sekunden):")
    print(tacq[:5])

except FileNotFoundError:
    print(f"Fehler: Die Datei '{file_path}' wurde nicht gefunden.")
except Exception as e:
    print(f"Ein Fehler ist aufgetreten: {e}")


Daten erfolgreich geladen!
Form des AIF-Arrays: (250,)
Form des Zeit-Arrays: (250,)

Erste 5 Werte des AIF:
[259.  269.1 255.2 240.8 257.5]

Erste 5 Werte der Zeitachse (in Sekunden):
[0.000000e+00 9.918213e-05 1.983643e-04 2.975464e-04 3.967285e-04]


In [17]:
aif_list = aif.tolist()
tacq_list = tacq.tolist()

In [18]:
# Make sure the folder exists before calling the function
if not os.path.exists(dicom_folder):
    os.makedirs(dicom_folder)
    print(f"Folder '{dicom_folder}' was created. Please fill it with your DICOM files.")
elif not os.listdir(dicom_folder):
    print(f"The folder '{dicom_folder}' is empty. Please add DICOM files to run the script.")
else:
    # Call the conversion function
    dicom_to_zarr.convert_dicom_to_zarr(dicom_folder, zarr_file)

Lese DICOM-Dateien aus: /mnt/dev_rep/repos/MRI-MoCoCo/ready_to_use/Raw_Datasets/159269_B1/14/DICOM
Stapele DICOM-Schichten zu einem 3D-Raum...
Extrahierte Metadaten: {'PixelSpacing': [1.9531, 1.9531], 'SliceThickness': 5.0, 'Rows': 256, 'Columns': 256, 'PatientID': 'MEDCIC_10_B1'}
Speichere Array der Größe (12500, 256, 256) in /mnt/dev_rep/repos/MRI-MoCoCo/ready_to_use/Raw_Datasets/159269_B1/14/159269_B1_14.zarr...
Konvertierung abgeschlossen!


In [19]:
data = zarr.open(zarr_file)

In [20]:
temp_zarr = 'input_zarr_for_mdreg_parallel_03.zarr'

In [65]:
# --- 1. Setup: Erstelle ein Dummy-Array mit Ihrer Start-Form ---
# Dies simuliert Ihr geladenes Zarr-Array.
input_array = da.from_zarr(data)

# --- 2. Definition der Zieldimensionen ---
H, W = 256, 256 # Höhe und Breite bleiben gleich
D = 50 # 50 Bilder pro Zeitaufnahme bilden die neue Tiefen-Dimension (Z-Richtung)
T = 250 # 250 zeitlich versetzte Aufnahmen

# Überprüfung: 50 * 250 muss 12500 ergeben.
assert D * T == input_array.shape[0], "Die Dimensionen passen nicht zusammen!"

# --- 3. Umformung und Neuordnung der Achsen ---

# Schritt A: Umformen (Reshape)
# Wir interpretieren die 12500 Bilder als 250 Zeitpunkte (T) mit je 50 Tiefenschichten (D).
# Das Ergebnis hat die Form (T, D, H, W).
reshaped_array = input_array.reshape(T, D, H, W)
print(f"Form nach dem Reshape: {reshaped_array.shape}  (entspricht T, D, H, W)")

# Schritt B: Achsen neuordnen (Transpose)
# Wir bringen die Achsen in die gewünschte Reihenfolge (H, W, D, T).
# Alte Achsen: 0=T, 1=D, 2=H, 3=W
# Neue Reihenfolge: 2, 3, 1, 0
final_array = reshaped_array.transpose(2, 3, 1, 0)
print(f"Form des finalen Arrays: {final_array.shape} (entspricht H, W, D, T)")

# --- 4. Überprüfung ---
# Das finale Array hat nun die gewünschte Form (256, 256, 50, 250).
# Sie können es jetzt z.B. als neues Zarr-Array speichern.
final_array.to_zarr(temp_zarr, overwrite=True)
correct_shape_array = zarr.open(temp_zarr)

Form nach dem Reshape: (250, 50, 256, 256)  (entspricht T, D, H, W)
Form des finalen Arrays: (256, 256, 50, 250) (entspricht H, W, D, T)


In [27]:
correct_shape_array

<Array file://input_zarr_for_mdreg_parallel_03.zarr shape=(256, 256, 50, 250) dtype=int16>

In [28]:
temp = correct_shape_array[:,:,25,:]
temp = np.transpose(temp, [2,0,1])

In [29]:
helpers.explore_3D_array(temp)

interactive(children=(IntSlider(value=124, description='SLICE', max=249), Output()), _dom_classes=('widget-int…

Model

In [30]:
class UNet3D(nn.Module):
    def __init__(self, in_channels=2, out_channels=3):
        super().__init__()
        self.enc1 = self._conv_block(in_channels, 16)
        self.enc2 = self._conv_block(16, 32)
        self.pool = nn.MaxPool3d(2)
        self.bottleneck = self._conv_block(32, 64)
        self.upconv2 = nn.ConvTranspose3d(64, 32, kernel_size=2, stride=2)
        self.dec2 = self._conv_block(64, 32)
        self.upconv1 = nn.ConvTranspose3d(32, 16, kernel_size=2, stride=2)
        self.dec1 = self._conv_block(32, 16)
        self.final_conv = nn.Conv3d(16, out_channels, kernel_size=1)

        self.final_conv.weight.data.zero_()
        self.final_conv.bias.data.zero_()

    def _conv_block(self, in_c, out_c):
        return nn.Sequential(
            nn.Conv3d(in_c, out_c, 3, 1, 1, bias=False),
            nn.InstanceNorm3d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_c, out_c, 3, 1, 1, bias=False),
            nn.InstanceNorm3d(out_c),
            nn.ReLU(inplace=True)
        )

    def forward(self, x_fixed, x_moving):
        x = torch.cat([x_fixed, x_moving], dim=1)
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        b = self.bottleneck(self.pool(e2))
        d2 = self.upconv2(b)
        
        # Sicherstellen, dass die Dimensionen für den Skip-Connection passen
        if d2.shape[2:] != e2.shape[2:]:
             d2 = F.interpolate(d2, size=e2.shape[2:], mode='trilinear', align_corners=False)

        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)
        d1 = self.upconv1(d2)

        if d1.shape[2:] != e1.shape[2:]:
            d1 = F.interpolate(d1, size=e1.shape[2:], mode='trilinear', align_corners=False)
            
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)
        return self.final_conv(d1)

# Standard-Implementierung des Spatial Transformers
class SpatialTransformer3D(nn.Module):
    def __init__(self, size):
        super(SpatialTransformer3D, self).__init__()
        self.size = size # Erwartete Größe: (D, H, W)
        vectors = [torch.arange(0, s) for s in size]
        grids = torch.meshgrid(vectors, indexing='ij')
        grid = torch.stack(grids).float()
        self.register_buffer('grid', grid.unsqueeze(0), persistent=False)

    def forward(self, src, flow):
        new_locs = self.grid + flow
        shape = flow.shape[2:]
        for i in range(len(shape)):
            new_locs[:, i, ...] = 2 * (new_locs[:, i, ...] / (shape[i] - 1) - 0.5)
        new_locs = new_locs.permute(0, 2, 3, 4, 1)
        new_locs = new_locs[..., [2, 1, 0]]
        return F.grid_sample(src, new_locs, align_corners=True, padding_mode="border")

# Asynchroner Worker-Thread zum Speichern der Ergebnisse
def save_to_zarr_worker(q, warped_zarr, dvf_zarr):
    while True:
        data = q.get()
        if data is None: break # Stopp-Signal
        time_idx, warped_np, dvf_np = data
        try:
            warped_zarr[..., time_idx] = warped_np
            dvf_zarr[..., time_idx, :] = dvf_np
        except Exception as e:
            print(f"Fehler im Writer-Thread beim Index {time_idx}: {e}")
        q.task_done()

In [31]:
class SelfRegistrationDataset(Dataset):
    def __init__(self, moving_path, fixed_index=0, k=4):
        print(f"Lade 4D Zarr-Array von: {moving_path}")
        self.moving_arr = zarr.open(moving_path, mode='r')
        self.fixed_index = fixed_index
        self.k = k

        # Annahme: Form ist (H, W, D, T)
        self.original_shape = self.moving_arr.shape[:-1]
        self.num_time_points = self.moving_arr.shape[-1]
        self.padded_shape = self._calculate_padded_shape(self.original_shape)

        print(f"Originale 3D-Form (H, W, D): {self.original_shape}")
        print(f"Gepaddete 3D-Form (teilbar durch {self.k}): {self.padded_shape}")

        print(f"Lade und bereite festes Referenzbild (Index {self.fixed_index}) vor...")
        fixed_np = self.moving_arr[..., self.fixed_index].astype(np.float32)
        fixed_tensor_unpadded = torch.from_numpy(fixed_np).permute(2, 0, 1).unsqueeze(0)
        self.fixed_tensor = self._pad_tensor(fixed_tensor_unpadded)

    def _calculate_padded_shape(self, shape): # (H, W, D)
        return tuple([(s + self.k - 1) // self.k * self.k for s in shape])

    def _pad_tensor(self, tensor): # (C, D, H, W)
        shape_in = tensor.shape
        shape_out = (1, self.padded_shape[2], self.padded_shape[0], self.padded_shape[1])
        pad_d = shape_out[1] - shape_in[1]
        pad_h = shape_out[2] - shape_in[2]
        pad_w = shape_out[3] - shape_in[3]
        padding = (pad_w // 2, pad_w - pad_w // 2, pad_h // 2, pad_h - pad_h // 2, pad_d // 2, pad_d - pad_d // 2)
        return F.pad(tensor, padding, "constant", 0)

    def __len__(self):
        return self.num_time_points

    def __getitem__(self, idx):
        moving_np = self.moving_arr[..., idx].astype(np.float32)
        moving_tensor_unpadded = torch.from_numpy(moving_np).permute(2, 0, 1).unsqueeze(0)
        moving_tensor = self._pad_tensor(moving_tensor_unpadded)
        return moving_tensor, self.fixed_tensor

In [76]:
# %% Zelle 4: Hauptskript (Konfiguration & Ausführung)
if __name__ == '__main__':
    # --- 1. Konfiguration ---
    MODEL_PATH = "/mnt/dev_rep/repos/MRI-MoCoCo/ready_to_use/best_supervised_model_250810_01_batch8_LR1e-4_Ep100_VSpl1_RegWeight01.pth" # BITTE ANPASSEN
    INPUT_ZARR_PATH = temp_zarr     # BITTE ANPASSEN
    OUTPUT_WARPED_PATH = "korrigierte_bilder.zarr"           # BITTE ANPASSEN
    OUTPUT_DVF_PATH = "displacement_fields.zarr"      # BITTE ANPASSEN
    
    FIXED_IMAGE_INDEX = 0  # Wählt das 2. Bild als festes Referenzbild
    NUM_WRITER_THREADS = 8 # Anzahl der Threads zum Speichern, je nach System anpassen

    # --- 2. Setup ---
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Verwende Gerät: {device}")
    
    for path in [OUTPUT_WARPED_PATH, OUTPUT_DVF_PATH]:
        if os.path.exists(path):
            if os.path.isdir(path): shutil.rmtree(path)
            else: os.remove(path)
            print(f"Warnung: Bestehendes Verzeichnis {path} wurde gelöscht.")

    # --- 3. Daten und Modell laden ---
    print("\nLade Datensatz...")
    dataset = SelfRegistrationDataset(INPUT_ZARR_PATH, fixed_index=FIXED_IMAGE_INDEX)
    
    padded_input_size = (dataset.padded_shape[2], dataset.padded_shape[0], dataset.padded_shape[1]) # (D, H, W)
    
    print("\nLade trainiertes Modell...")
    model = UNet3D().to(device)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    model.eval()
    print("Modell-Gewichte erfolgreich geladen.")
    
    stn = SpatialTransformer3D(size=padded_input_size).to(device)
    
    dataloader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=max(1, os.cpu_count() // 2), pin_memory=True, prefetch_factor=2)

    # --- 4. Output-Dateien und Writer-Threads erstellen ---
    print("\nErstelle thread-sichere Ausgabedateien...")
    synchronizer = threading.Lock()
    
    warped_output_zarr = zarr.open(OUTPUT_WARPED_PATH, mode='w', shape=dataset.moving_arr.shape, chunks=dataset.moving_arr.chunks, dtype=dataset.moving_arr.dtype, compressor=None, synchronizer=synchronizer, zarr_version=2)
    
    dvf_shape = dataset.moving_arr.shape + (3,)
    dvf_chunks = dataset.moving_arr.chunks + (3,)
    dvf_output_zarr = zarr.open(OUTPUT_DVF_PATH, mode='w', shape=dvf_shape, chunks=dvf_chunks, dtype='float32', compressor=None, synchronizer=synchronizer, zarr_version=2)
    
    data_queue = queue.Queue(maxsize=NUM_WRITER_THREADS * 8)
    writer_threads = []
    print(f"Starte {NUM_WRITER_THREADS} asynchrone Writer-Threads...")
    for _ in range(NUM_WRITER_THREADS):
        thread = threading.Thread(target=save_to_zarr_worker, args=(data_queue, warped_output_zarr, dvf_output_zarr))
        thread.daemon = True
        thread.start(); writer_threads.append(thread)

    # --- 5. Inferenz-Schleife ---
    print(f"\nStarte Inferenz für {len(dataset)} Volumen...")
    original_shape_dims = dataset.original_shape # (H, W, D)
    padded_shape_dims = dataset.padded_shape   # (H_pad, W_pad, D_pad)
    
    with torch.no_grad():
        for i, (moving_batch, fixed_batch) in enumerate(tqdm.tqdm(dataloader, desc="Verarbeite Datensatz")):
            moving_batch = moving_batch.to(device, non_blocking=True); fixed_batch = fixed_batch.to(device, non_blocking=True)
            
            # Autocast für schnellere Inferenz mit float16 auf GPUs
            with torch.autocast(device_type=str(device), dtype=torch.float16):
                predicted_dvf_batch = model(fixed_batch, moving_batch)
                warped_batch = stn(moving_batch, predicted_dvf_batch)
            
            # Ergebnisse auf CPU bringen und Padding entfernen
            warped_tensor = warped_batch.squeeze().cpu(); disp_field = predicted_dvf_batch.squeeze().cpu()
            
            pad_h_start = (padded_shape_dims[0] - original_shape_dims[0]) // 2
            pad_w_start = (padded_shape_dims[1] - original_shape_dims[1]) // 2
            pad_d_start = (padded_shape_dims[2] - original_shape_dims[2]) // 2
            
            cropped_warped = warped_tensor[pad_d_start:pad_d_start+original_shape_dims[2], 
                                           pad_h_start:pad_h_start+original_shape_dims[0], 
                                           pad_w_start:pad_w_start+original_shape_dims[1]]
            
            cropped_disp = disp_field[:, 
                                      pad_d_start:pad_d_start+original_shape_dims[2], 
                                      pad_h_start:pad_h_start+original_shape_dims[0], 
                                      pad_w_start:pad_w_start+original_shape_dims[1]]
            
            # In die Speicher-Form (H, W, D) umwandeln
            warped_np = cropped_warped.numpy().transpose(1, 2, 0)
            # In die Speicher-Form (H, W, D, 3) umwandeln
            dvf_np = cropped_disp.numpy().transpose(2, 3, 1, 0)
            
            data_queue.put((i, warped_np, dvf_np))
            
    # --- 6. Aufräumen ---
    print("\nHauptprozess abgeschlossen. Sende Stopp-Signal an Writer-Threads...")
    for _ in range(NUM_WRITER_THREADS): data_queue.put(None)
    
    print("Warte, bis alle Writer-Threads ihre Arbeit beendet haben...")
    for thread in writer_threads: thread.join()

    print(f"\nVerarbeitung und Speicherung abgeschlossen.")
    print(f"Korrigierte Bilder: {OUTPUT_WARPED_PATH}")
    print(f"Displacement Fields: {OUTPUT_DVF_PATH}")

Verwende Gerät: cuda
Warnung: Bestehendes Verzeichnis korrigierte_bilder.zarr wurde gelöscht.


/home/shooty/anaconda3/envs/ds-env-01/lib/python3.13/site-packages/zarr/api/asynchronous.py:1267: RuntimeWarning: synchronizer is not yet implemented
  return await create(


Warnung: Bestehendes Verzeichnis displacement_fields.zarr wurde gelöscht.

Lade Datensatz...
Lade 4D Zarr-Array von: input_zarr_for_mdreg_parallel_03.zarr
Originale 3D-Form (H, W, D): (256, 256, 50)
Gepaddete 3D-Form (teilbar durch 4): (256, 256, 52)
Lade und bereite festes Referenzbild (Index 0) vor...

Lade trainiertes Modell...
Modell-Gewichte erfolgreich geladen.

Erstelle thread-sichere Ausgabedateien...
Starte 8 asynchrone Writer-Threads...

Starte Inferenz für 250 Volumen...


Verarbeite Datensatz: 100%|██████████| 250/250 [00:30<00:00,  8.28it/s]



Hauptprozess abgeschlossen. Sende Stopp-Signal an Writer-Threads...
Warte, bis alle Writer-Threads ihre Arbeit beendet haben...

Verarbeitung und Speicherung abgeschlossen.
Korrigierte Bilder: korrigierte_bilder.zarr
Displacement Fields: displacement_fields.zarr


In [77]:
coreg = zarr.open(OUTPUT_WARPED_PATH)

In [78]:
coreg

<Array file://korrigierte_bilder.zarr shape=(256, 256, 50, 250) dtype=int16>

In [79]:
correct_shape_array

<Array file://input_zarr_for_mdreg_parallel_03.zarr shape=(256, 256, 50, 250) dtype=int16>

In [80]:
coreg_vis = coreg [:,:,25,:]
coreg_vis = np.transpose(coreg_vis, [2,0,1])

In [81]:
helpers.explore_3D_array(coreg_vis)

interactive(children=(IntSlider(value=124, description='SLICE', max=249), Output()), _dom_classes=('widget-int…

In [82]:
pars

NameError: name 'pars' is not defined

In [83]:
fit_vis = fit [:,:,25,:]
fit_vis = np.transpose(fit_vis, [2,0,1])

NameError: name 'fit' is not defined

In [84]:
helpers.explore_3D_array(fit_vis)

NameError: name 'fit_vis' is not defined

In [85]:
correct_shape_array


<Array file://input_zarr_for_mdreg_parallel_03.zarr shape=(256, 256, 50, 250) dtype=int16>

In [86]:
correct_shape = correct_shape_array[:,:,25,:]
correct_shape = np.transpose(correct_shape,[2,0,1] ) 

In [87]:
helpers.explore_3D_array_comparison_and_diff(coreg_vis, correct_shape)

interactive(children=(IntSlider(value=125, description='Slice:', max=249), Output()), _dom_classes=('widget-in…

In [ ]:
t0, t1 = 0,249

In [ ]:
coreg_rotate = np.transpose(coreg, [1,0,2,3])

In [ ]:
anim = mdreg.plot.animation(
    coreg_rotate[:,:,:, t0:t1],
    title=f"{DATASET_NAME}_{PART} Coreg",
    vmin=0,
    vmax=0.9*np.max(coreg[...,0],)
)

In [ ]:
import matplotlib.animation as animation

writervideo = animation.FFMpegWriter(fps=15)
anim.save(f'{DATASET_NAME}_{PART}_4.mp4', writer=writervideo)

#anim.save(f'{DATASET_NAME}_{PART}_Coreg.mpeg')